In [2]:
%load_ext autoreload
%autoreload 2
import pyarrow as pa
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.retriever import ComplaintRetriever
from src.prod_vector_store import build_vector_store,load_vector_store
from src.generator import ComplaintGenerator
from src.rag import ComplaintRAG



# Build Vector Store

In [2]:
collection = build_vector_store(
    parquet_path="../data/raw/complaint_embeddings.parquet",
    persist_directory="../data/vector_store",
    collection_name="complaints"
)

Loaded 1,375,327 records


Indexing Chunks:   0%|          | 0/252 [00:00<?, ?it/s]

Successfully indexed 1,375,327 chunks


# Load Vector Store

In [3]:
collection = load_vector_store(
    persist_directory="../data/vector_store",
    collection_name="complaints"
)

Loaded collection with 1,375,327 chunks


# Test Retriever

In [4]:
retriever = ComplaintRetriever(
    persist_directory="../data/vector_store"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Ask a question

In [8]:
results = retriever.retrieve(
    "Why are customers unhappy with credit cards?",
    k=5
)

In [9]:
for i, doc in enumerate(
    results["documents"][0],
    start=1
):

    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)

    print(doc[:1000])

    print("\n")

Result 1
card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them. i am not happy with the customer service and i am not happy with the misinformation i was given. i have been given misinformation by several customer service representatives and i feel that the credit card company is taking advantage of consumers who dont understand the features and benefits.


Result 2
creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my credit usage shoot up. what the is up with these companies it like here is the card but don't use it!


Result 3
credit card companies think of their card holders. the indifferent response to a long-term card holder again came to me as a shock given how this particular company was receptive to me in the past when i called about other

In [49]:
from src.rag import ComplaintRAG

rag = ComplaintRAG()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [50]:
result = rag.ask(
    "Why are customers unhappy with credit cards?"
)

print(result["answer"])

Customers are unhappy with credit cards due to several reasons highlighted in the complaints:

1. **Misinformation and Poor Customer Service:** Customers feel they were given incorrect information by customer service representatives, leading to frustration and a lack of trust in the credit card company's communication.

2. **Unjustified Credit Limit Reductions:** There are complaints about credit limits being reduced without valid reasons, which can lead to higher credit utilization ratios and potential financial strain for the customers.

3. **Lack of Loyalty and Support:** Despite having a good payment history, customers feel that credit card companies do not show loyalty or support to long-term cardholders, which was unexpected and disappointing.

4. **Unexpected Changes in Interest Rates:** Customers are unhappy with sudden increases in interest rates, making it more difficult to manage and pay off their credit card debts.

5. **Overall Negative Perception:** The complaints reflect

# RAG Evaluation

In [42]:
evaluation_questions = [
    "Why are customers unhappy with credit cards?",
    "What are the most common complaints about personal loans?",
    "Why are customers dissatisfied with savings accounts?",
    "What issues do customers report when using money transfer services?",
    "What problems do customers experience during credit card applications?",
    "What account management issues are frequently reported by credit card customers?",
    "Are there recurring complaints related to unexpected fees or charges?",
    "What customer service issues appear across multiple financial products?",
    "What are the main causes of disputes between customers and financial institutions?",
    "What emerging complaint trends should product managers monitor?"
]

In [43]:
results = []

for question in evaluation_questions:
    
    response = rag.ask(question)

    results.append({
        "Question": question,
        "Answer": response["answer"],
        "Sources": response["sources"][:2]  # first 2 sources
    })

print(f"Evaluated {len(results)} questions.")

Evaluated 10 questions.


In [44]:
for result in results:
    
    print("=" * 100)
    print("QUESTION:")
    print(result["Question"])

    print("\nANSWER:")
    print(result["Answer"])

    print("\nSOURCES:")
    for source in result["Sources"]:
        print(
            f"- {source['product_category']} | "
            f"{source['issue']} | "
            f"{source['company']}"
        )

    print("\n")

QUESTION:
Why are customers unhappy with credit cards?

ANSWER:
Customers are unhappy with credit cards due to several reasons highlighted in the complaints:

1. **Misinformation and Poor Customer Service:** Customers feel they were given incorrect information by customer service representatives, leading to frustration and a perception that the credit card company is taking advantage of consumers who do not fully understand the features and benefits.

2. **Unjustified Credit Limit Reductions:** There are complaints about credit card companies reducing credit limits without valid reasons, which can increase credit usage and cause financial harm to the customers.

3. **Lack of Loyalty:** Customers express disappointment in the lack of loyalty from credit card companies, noting that long-term use of a card does not guarantee better treatment or support from the company.

4. **Unexpected Changes in Interest Rates:** Customers are unhappy with sudden increases in interest rates, which make 

In [45]:
import pandas as pd

evaluation_df = pd.DataFrame({
    "Question": [r["Question"] for r in results],
    "Generated Answer": [r["Answer"] for r in results],
    "Sources": [r["Sources"] for r in results],
    "Quality Score": [""] * len(results),
    "Comments/Analysis": [""] * len(results)
})

evaluation_df

,Question,Generated Answer,Sources,Quality Score,Comments/Analysis
0,Why are customers unhappy with credit cards?,Customers are unhappy with credit cards due to...,"[{'issue': 'Other features, terms, or problems...",,
1,What are the most common complaints about pers...,The most common complaints about personal loan...,"[{'date_received': '2020-10-21', 'state': 'CA'...",,
2,Why are customers dissatisfied with savings ac...,Customers are dissatisfied with savings accoun...,"[{'state': 'VA', 'chunk_index': 4, 'date_recei...",,
3,What issues do customers report when using mon...,Customers report several issues when using mon...,"[{'complaint_id': '2765911', 'issue': 'Other t...",,
4,What problems do customers experience during c...,I do not have enough information from the retr...,"[{'company': 'BANK OF AMERICA, NATIONAL ASSOCI...",,
5,What account management issues are frequently ...,"Based on the provided context, the account man...","[{'product_category': 'Credit Card', 'state': ...",,
6,Are there recurring complaints related to unex...,"Yes, there are recurring complaints related to...","[{'state': 'CA', 'total_chunks': 3, 'complaint...",,
7,What customer service issues appear across mul...,"Based on the provided context, the customer se...","[{'complaint_id': '5294876', 'company': 'JPMOR...",,
8,What are the main causes of disputes between c...,"Based on the provided context, the main causes...","[{'total_chunks': 5, 'company': 'U.S. BANCORP'...",,
9,What emerging complaint trends should product ...,I do not have enough information from the retr...,"[{'date_received': '2019-03-19', 'chunk_index'...",,


In [47]:
evaluation_df.to_csv("../data/processed/evaluation.csv", index=False)